In [ ]:
# 03_violation_baseline.ipynb -- LightGBM baseline for the 1-4-slot-lead-time
# frequency-violation target
# !pip install lightgbm -q

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve

import features as f

TARGET = "violation_lead"

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
feat_df = f.build_feature_table(scada)

df = feat_df.dropna(subset=[TARGET]).copy()  # drop rows whose lead window is unresolvable

# --- Time-aware split ---
train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
test = df[df["date"] >= "2026-01-01"]

print("train:", train.shape, "val:", val.shape, "test:", test.shape)
print("event rate train/val/test:", train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean())

X_train, y_train = train[f.FEATURE_COLS], train[TARGET]
X_val, y_val = val[f.FEATURE_COLS], val[TARGET]
X_test, y_test = test[f.FEATURE_COLS], test[TARGET]

# NOTE (2026-07-11): deliberately NOT using scale_pos_weight here -- see
# features.py's scale_pos_weight() docstring for the full story. In short: it
# caused LightGBM's early stopping to fire after a single boosting round
# (best_iteration_=1) for this ~2-3% positive-rate target, in every configuration
# tried, silently training something close to a single shallow tree instead of
# the intended up-to-500-round ensemble. Removing it and using average_precision
# (not the default binary_logloss) as the early-stopping metric fixed this.
model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)
print(f"\nbest_iteration_: {model.best_iteration_}  (sanity check -- should NOT be 1)")

proba_test = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, proba_test)
base_rate = y_test.mean()
print(f"PR-AUC: {pr_auc:.4f}  (random baseline = base rate = {base_rate:.4f})")

preds_05 = (proba_test >= 0.5).astype(int)
print(f"F1 @ 0.5 threshold: {f1_score(y_test, preds_05):.4f}")

# A fixed 0.5 threshold is the wrong lens for a ~2-3% positive rate -- it almost never
# fires. Report the best-F1 operating point too, which is what an actual early-warning
# system would be tuned to.
precision, recall, thresh = precision_recall_curve(y_test, proba_test)
f1s = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = np.nanargmax(f1s[:-1])
print(f"Best-F1 operating point: F1={f1s[best_idx]:.4f} at threshold={thresh[best_idx]:.4f} "
      f"(precision={precision[best_idx]:.4f}, recall={recall[best_idx]:.4f})")

idx95 = np.where(precision[:-1] >= 0.95)[0]
recall_at_95p = recall[idx95].max() if len(idx95) else 0.0
print(f"Recall at >=95% precision: {recall_at_95p:.4f}")

importance = pd.Series(model.feature_importances_, index=f.FEATURE_COLS).sort_values(ascending=False)
print("\nTop 15 features:\n", importance.head(15))

# --- Results (verified 2026-07-11, LightGBM 4.6.0; third version of this notebook's
#     results, after two rounds of real fixes -- see below) ---
# best_iteration_: 70 (not 1 -- confirms the model is actually training, not stalling)
# PR-AUC 0.1186 vs a random/base-rate baseline of 0.0305 -- 3.88x lift over chance.
# F1@0.5 near-zero still (0.5 is the wrong threshold for a ~3% positive rate). Best-F1
# operating point: F1=0.1852, precision=13.6%, recall=29.0%. Recall at >=95% precision
# is 0.0036 -- effectively zero, not a usable operating point, but worth stating
# precisely rather than rounding down to "0" and overstating how flat that number is.
#
# History of this notebook's numbers, since the trail matters more than any single
# snapshot: (1) original PR-AUC 0.0614 -- scale_pos_weight was silently limiting
# training to 1 boosting round; (2) fixing that + adding solar-volatility features
# (solar_delta_mw, solar_roll8_std, after the two-stage ramp->violation hypothesis was
# tested and found false) raised it to PR-AUC 0.0937; (3) THIS version: removing
# share_res_pct and the 11 corridor/cross-border columns from the feature set raised it
# further to 0.1186. That removal wasn't originally planned as an improvement -- it
# started as a leakage check. share_res_pct and every ir_*/xb_* column are whole-DAY
# aggregates broadcast identically to all 96 slots of a day (verified directly: every
# row of a given date has the exact same value, unlike freq_hz/demand_met_mw which
# genuinely vary per slot) -- raising the question of whether they leak later-in-day
# information into an earlier slot's prediction. Tested rather than assumed: removing
# them IMPROVED the model rather than degrading it, meaning there was no leakage-driven
# inflation to worry about, and these columns were pure noise once real per-slot signals
# (frequency/demand lags, solar volatility, hour) are available. Not deleted from
# features.py -- see DAILY_BROADCAST_COLS there -- since Era 2's daily-resolution
# corridor-flow finding is a separate, valid analysis unaffected by this classifier-level
# result.


In [ ]:
# --- Appendix: shorter lead-window experiment (2026-07-11, re-verified a second time
#     after removing share_res_pct/corridor columns from FEATURE_COLS -- see the main
#     cell above) ---
# Does shrinking the lookahead window from 1-4 slots improve the violation classifier?
# features.py's add_violation_label() and build_feature_table() both take a lead_slots
# override for exactly this test.

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import average_precision_score, precision_recall_curve

import features as f

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
resid = f.build_study1_residual_signal()  # computed once, reused across all window sizes


def run(lead_slots):
    feat = f.build_feature_table(scada, study1_residual=resid, violation_lead_slots=lead_slots)
    df = feat.dropna(subset=["violation_lead"]).copy()
    train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
    val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
    test = df[df["date"] >= "2026-01-01"]

    X_train, y_train = train[f.FEATURE_COLS], train["violation_lead"]
    X_val, y_val = val[f.FEATURE_COLS], val["violation_lead"]
    X_test, y_test = test[f.FEATURE_COLS], test["violation_lead"]

    model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="average_precision",
              callbacks=[lgb.early_stopping(50, verbose=False)])

    proba = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, proba)
    base_rate = y_test.mean()

    precision, recall, thresh = precision_recall_curve(y_test, proba)
    f1s = 2 * precision * recall / (precision + recall + 1e-12)
    best_idx = np.nanargmax(f1s[:-1])

    print(f"lead_slots={lead_slots}: best_iter={model.best_iteration_}  base_rate={base_rate:.4f}  "
          f"PR-AUC={pr_auc:.4f} (lift={pr_auc / base_rate:.2f}x)  best-F1={f1s[best_idx]:.4f} "
          f"(P={precision[best_idx]:.4f} R={recall[best_idx]:.4f})")


for k in [4, 3, 2, 1]:
    run(k)

# --- Findings (re-verified 2026-07-11, third pass -- these numbers supersede both
#     earlier versions of this appendix) ---
# lead_slots=4 (shipped, 15-60 min): PR-AUC=0.1186 (3.88x lift)  best-F1=0.1852 (P=13.6% R=29.0%)
# lead_slots=3 (15-45 min):          PR-AUC=0.0951 (3.86x lift)  best-F1=0.1609 (P=11.9% R=24.8%)
# lead_slots=2 (15-30 min):          PR-AUC=0.1329 (7.33x lift)  best-F1=0.1767 (P=17.1% R=18.4%)
# lead_slots=1 (15 min only):        PR-AUC=0.1129 (10.18x lift) best-F1=0.2228 (P=24.4% R=20.5%)
#
# Worth being explicit about: this is the THIRD time this exact experiment has been run,
# each time on a meaningfully different (and better) feature set, and each time the
# "best" window has moved -- first no clean winner, then lead_slots=2 looked clearly
# best on every metric, now lead_slots=1 has the best F1 while lead_slots=2 still has
# the best PR-AUC and lead_slots=4 remains competitive on PR-AUC too. That instability
# is itself informative: whatever the "optimal" window is, it isn't a robust, clearly
# separated winner -- it's sensitive to modelling choices in a way that argues for
# NOT over-indexing on any single run's ranking. The one consistent pattern across all
# three passes: shorter windows tend to trade recall for precision/F1, longer windows
# tend to trade precision for recall and more total events caught. That qualitative
# trade-off is the reliable takeaway; the exact numbers per window are not stable enough
# to treat as a final answer. The shipped default in predict.py remains 4 slots
# (matching the roadmap's original 1-4-slot scope) -- changing it would need a real
# product decision plus probably more robust validation (e.g. repeated splits or
# cross-validation) than a single time-aware train/val/test split can offer, given how
# much these numbers move between otherwise-reasonable configurations.
